In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import time
from birddog.core import (
    Archive,
    ArchiveWatcher,
    )
from birddog.wiki import get_all_pages, mw_read_page, canonicalize_title
from birddog.runtime import Runtime

2025-07-13 17:26:36,829 [INFO] Using Google Cloud translation API (credentials file:/Users/jbrandt/code/birddog/google-cloud-translate-key.json)
2025-07-13 17:26:36,833 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.


In [3]:
titles = [ 
    "Архів:ДАПО/Р-9126/2",
    "Архів:ДАЖО/1/1",
    "Архів:ДАК/312/1",
    "Архів:ДАЖО/1/74",
    "Архів:Лука_Мала",
    "Архів:ЦДІАК/1/1",
    "Архів:ДАХО/Д",
    "Архів:ДАПО/Р/1–1000",
    "Архів:ЦДІАК/28",
    "Архів:ЦДІАК/28/1",
    "Архів:ДАЖО/1",
    "Архів:ДАК/Р-352",
    "Архів:Архівний відділ виконавчого комітету Кременчуцької міської ради/Р",
    "Архів:ДАЖО/Д",
    "Архів:ДАДнО/Р-6478/2", 
    "Архів:ДАЖО/752", 
    "Архів:ДАКрО/225/1/25", 
    "Архів:ДАКрО/225", 
    "Архів:ДАСО/Р", 
    "Архів:ДАХмО/К", 
    "Архів:ДАКрО/225/1/144а", 
    "Архів:ДАПО/978/1",
    "Архів:ДАПО/Р",
    "Архів:ДАХмО/Р-6193",
    "Архів:ДАКрО/П-5907/2Р",
    "Архів:ДАОО/Р-8085/1",
    "Архів:ДАКрО/225/1",
    "Архів:ДАПО/1072/1/1",
    "Архів:ДАПО/978/1/135",
    "Архів:ДАПО/978",
    "Архів:ДАКО/Р-5634/1/3092",
    ]

In [4]:
runtime = Runtime()

In [ ]:
address = runtime.lookup_address("Архів:ДАПО/Р/9126/2")
print(address)

In [ ]:
runtime.lookup(*address)
print(address)

In [ ]:
runtime.lookup(*address).name

In [ ]:
None in list(range(10))

In [ ]:
None in []

In [6]:
for title in titles:
    address = runtime.lookup_address(title)
    page = runtime.lookup(*address)
    print(f"{title}: {address}, {page.report}")
    assert canonicalize_title(title) == canonicalize_title(page.title)

Архів:ДАПО/Р-9126/2: ('DAPO', 'R', 'Р-9126', '2', ''), opus,DAPO-R/Р-9126/2,2025,06,15,15:22
Архів:ДАЖО/1/1: ('DAZHO', 'D', '1', '1', ''), opus,DAZHO-D/1/1,2025,04,29,19:30
Архів:ДАК/312/1: ('DAK', 'D', '312', '1', ''), opus,DAK-D/312/1,2025,06,26,08:44
Архів:ДАЖО/1/74: ('DAZHO', 'D', '1', '74', ''), opus,DAZHO-D/1/74,2025,05,31,19:05
Архів:Лука_Мала: ['Decerkva', 'MalaLuka', '', '', ''], archive,Decerkva-MalaLuka,2024,08,02,19:08
2025-07-13 17:27:21,499 [INFO] TitleIndex.lookup(Архів:ЦДІАК/1/1)
2025-07-13 17:27:21,716 [INFO] fetch_url: 14 requests in last 60s → 0.23 req/s
Архів:ЦДІАК/1/1: ('CDIAK', '_', '1', '1', ''), opus,CDIAK-_/1/1,2024,06,09,15:50
Архів:ДАХО/Д: ['DAHO', 'D', '', '', ''], archive,DAHO-D,2024,07,27,20:24
2025-07-13 17:27:22,402 [INFO] TitleIndex.lookup(Архів:ДАПО/Р/1–1000)
Архів:ДАПО/Р/1–1000: ('DAPO', 'R', '1–1000', '', ''), opus,DAPO-R/1–1000,2021,03,20,14:42
2025-07-13 17:27:22,612 [INFO] TitleIndex.lookup(Архів:ЦДІАК/28)
Архів:ЦДІАК/28: ('CDIAK', '_', '28', '', 

In [ ]:
def select_parent_archive(archive_root, fond_id, ti=manager._title_index):
    parent_archive = None
    known_child = False
    if not archive_root in ti._archives:
        raise ValueError(f"Unknown archive root: {archive_root}")
    for archive_address in ti._archives[archive_root]:
        archive = ti._lru.lookup(*archive_address)
        #print(archive.title)
        if fond_id.upper().startswith(archive.subarchive["uk"]):
            parent_archive = archive_address
            known_child = fond_id in archive.child_ids
            break
        elif archive_address[1] == "D" or len(ti._archives[archive_root]) == 1:
            parent_archive = archive_address
            known_child = fond_id in archive.child_ids
    return parent_archive, known_child

In [ ]:
def test_title(title, ti=manager._title_index):
    try:
        address = ti.lookup(title)
        return "known"
        #print("known:", title, address)
    except ValueError:
        title = canonicalize_title(title)
        title_split = title.split("/")
        if len(title_split) > 1:
            try:
                parent_archive, known_fond = select_parent_archive(*title_split[:2], ti)
                if not known_fond:
                    #print("need to adopt:", parent_archive, title)
                    return "adopt"
                else:
                    #print("something unexpected:", title)
                    return "unexpected"
            except ValueError:
                #print("unknown archive root:", title)
                return "unknown_archive"
        else:
            #print("unknown archive:", title)
            return "unknown_archive"

In [ ]:
test_title("Архів:ДАЧкО/5899/1")

In [ ]:
adoptees = [title for title in manager._tracker._mod_dates.keys() if test_title(title) == "adopt"]

In [ ]:
len(adoptees)

In [ ]:
adoptees[:20]

In [ ]:
test_result = {title: test_title(title) for title in manager._tracker._mod_dates.keys()}

In [ ]:
[title for title, result in test_result.items() if result == "unknown_archive"]

In [ ]:
orphans = []
for adoptee in adoptees:
    title_split = adoptee.split("/")
    if len(title_split) > 2:
        if test_title("/".join(title_split[:2])) != "adopt":
            orphans.append(adoptee)

In [ ]:
items = manager._tracker._mod_dates

In [ ]:
len(items.keys())

In [ ]:
suspects = [k for k in items.keys() if k.startswith("Архів:ДАПО") and "9126" in k and "Р-9126" not in k]

In [ ]:
from birddog.wiki import batch_page_exists

In [ ]:
batch_page_exists(suspects)